# Disease Risk Prediction from Medical Data
### Heart disease classification on the UCI Cleveland dataset

This notebook is the readable walk-through of the project. It re-uses the same
modules the application runs on (`src/`), so nothing here is a re-implementation
that could drift from what is deployed.

* Full training run: `python -m src.train`
* Web application:   `python -m uvicorn app.main:app --port 8000`

Every number below is computed when the notebook runs.

In [1]:
import json, sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd

from src import config as C
from src.data_loader import load_clean, profile, feature_dictionary, split_xy

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 30)

print("dataset :", C.DATASET_NAME)
print("source  :", C.DATASET_PAGE)
print("seed    :", C.RANDOM_SEED, "| test size:", C.TEST_SIZE)

dataset : UCI Heart Disease (Cleveland)
source  : https://archive.ics.uci.edu/dataset/45/heart+disease
seed    : 42 | test size: 0.2


## 1. Dataset selection

Three candidates were profiled before choosing (`scripts/dataset_survey.py`):

| Candidate | Rows x features | Minority class | Missing | Feature types |
|---|---|---|---|---|
| **UCI Heart Disease (Cleveland)** | 303 x 13 | 45.9% | 6 cells in 2 columns | 8 categorical-like + 5 continuous |
| Breast Cancer Wisconsin | 569 x 30 | 37.3% | none | 30 continuous |
| Pima Indians Diabetes | 768 x 8 | 34.9% | 374 hidden zeros in `insulin` alone | 8 continuous |

Cleveland was chosen because it is the only one of the three that:

* covers all four feature groups the brief asks for - symptoms, age, blood
  tests and clinical measurements;
* mixes categorical and continuous inputs, so a `ColumnTransformer` is genuinely
  needed rather than decorative;
* carries real, documented missing values, so imputation has to be handled
  inside the pipeline;
* has clinically named columns that can be shown to a person in a form. Breast
  Cancer's "worst fractal dimension" cannot.

Its weakness is size: 303 rows is small, and every interval below is wide
because of it.

In [2]:
df, cleaning = load_clean()
prof = profile(df)

print(f"rows x cols     : {prof['n_rows']} x {prof['n_cols']}")
print(f"target counts   : {prof['target_counts']}")
print(f"class share     : {prof['target_share']}")
print(f"imbalance ratio : {prof['imbalance_ratio']}:1")
print(f"duplicate rows  : {prof['duplicate_rows']}")
print(f"missing cells   : {sum(prof['missing_per_column'].values())}")
df.head()

rows x cols     : 303 x 14
target counts   : {0: 164, 1: 139}
class share     : {0: 0.5413, 1: 0.4587}
imbalance ratio : 1.18:1
duplicate rows  : 0
missing cells   : 6


,age,sex,cp,exang,trestbps,chol,fbs,restecg,thalach,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,0.0,145.0,233.0,1.0,2.0,150.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,1.0,160.0,286.0,0.0,2.0,108.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,1.0,120.0,229.0,0.0,2.0,129.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,0.0,130.0,250.0,0.0,0.0,187.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,0.0,130.0,204.0,0.0,2.0,172.0,1.4,1.0,0.0,3.0,0


## 2. Feature dictionary

Rendered from the schema in `src/config.py`, which is also what generates the web form.

In [3]:
pd.DataFrame(feature_dictionary())[
    ["feature", "ui_label", "type", "units", "domain", "group"]
]

,feature,ui_label,type,units,domain,group
0,age,Age,numeric,years,18 - 100,Patient Information
1,sex,Sex,binary,-,"0 = Female, 1 = Male",Patient Information
2,cp,Chest Pain Type,nominal,-,"1 = Typical angina, 2 = Atypical angina, 3 = N...",Symptoms
3,exang,Exercise-Induced Angina,binary,-,"0 = No, 1 = Yes",Symptoms
4,trestbps,Resting Blood Pressure,numeric,mm Hg,80 - 220,Vitals & Blood Tests
5,chol,Serum Cholesterol,numeric,mg/dl,100 - 600,Vitals & Blood Tests
6,fbs,Fasting Blood Sugar above 120 mg/dl,binary,-,"0 = No, 1 = Yes",Vitals & Blood Tests
7,restecg,Resting ECG Result,nominal,-,"0 = Normal, 1 = ST-T wave abnormality, 2 = Lef...",ECG & Exercise Test
8,thalach,Maximum Heart Rate Achieved,numeric,bpm,60 - 220,ECG & Exercise Test
9,oldpeak,ST Depression (Exercise vs Rest),numeric,mm,0 - 7,ECG & Exercise Test


In [4]:
stats = pd.DataFrame(prof["columns"]).T[["dtype", "missing", "n_unique", "min", "max", "mean", "std"]]
stats.loc[C.FEATURE_ORDER]

,dtype,missing,n_unique,min,max,mean,std
age,float64,0,41,29.0,77.0,54.4389,9.0387
sex,float64,0,2,0.0,1.0,0.6799,0.4673
cp,float64,0,4,1.0,4.0,3.1584,0.9601
exang,float64,0,2,0.0,1.0,0.3267,0.4698
trestbps,float64,0,50,94.0,200.0,131.6898,17.5997
chol,float64,0,152,126.0,564.0,246.6931,51.7769
fbs,float64,0,2,0.0,1.0,0.1485,0.3562
restecg,float64,0,3,0.0,2.0,0.9901,0.995
thalach,float64,0,91,71.0,202.0,149.6073,22.875
oldpeak,float64,0,40,0.0,6.2,1.0396,1.1611


## 3. Cleaning decisions

Each decision, and the reason for it, is recorded by the loader rather than
described after the fact.

In [5]:
for i, note in enumerate(cleaning.notes, 1):
    print(f"{i}. {note}\n")

1. Target: 'num' (0-4 severity) binarised to 0 / 1. Values 1-4 all mean >50% narrowing in at least one vessel; the 1-4 sub-levels have very few samples each (n<40), so a 5-class model would be unreliable at n=303.

2. Missing values found in ['ca', 'thal'] (6 cells, 0.14% of the frame). They are NOT dropped here: imputation happens inside the modelling pipeline so it is fitted on training folds only (leakage control).

3. Exact duplicate rows: 0. None found, nothing removed.

4. No value fell outside its documented UCI domain, so no coercion was needed.

5. Outliers retained. Extreme values here (e.g. chol = 564 mg/dl) are clinically plausible and informative; deleting them at n=303 would discard real signal. Robustness is instead handled by scaling and by regularised / tree-based models.



## 4. Exploratory analysis

Figures are written to `outputs/figures/`. The numbers behind them are below.

In [6]:
from src.eda import run_eda
eda = run_eda(df)

print("Correlation with the outcome, strongest first:")
for k, v in eda["insights"]["target_correlation"].items():
    print(f"  {C.FEATURE_BY_NAME[k]['label']:<40} {v:+.3f}")

Correlation with the outcome, strongest first:
  Thallium Stress Test Result              +0.526
  Major Vessels Coloured by Fluoroscopy    +0.460
  Exercise-Induced Angina                  +0.432
  ST Depression (Exercise vs Rest)         +0.425
  Maximum Heart Rate Achieved              -0.417
  Chest Pain Type                          +0.414
  Peak Exercise ST Segment Slope           +0.339
  Sex                                      +0.277
  Age                                      +0.223
  Resting ECG Result                       +0.169
  Resting Blood Pressure                   +0.151
  Serum Cholesterol                        +0.085
  Fasting Blood Sugar above 120 mg/dl      +0.025


In [7]:
print("Standardised mean difference (Cohen's d) between outcome groups:")
for k, v in eda["insights"]["numeric_separation"].items():
    print(f"  {C.FEATURE_BY_NAME[k]['label']:<40} d = {v['cohens_d']:+.3f}"
          f"   (neg {v['mean_negative']}, pos {v['mean_positive']})")

Standardised mean difference (Cohen's d) between outcome groups:
  Age                                      d = +0.461   (neg 52.59, pos 56.63)
  Resting Blood Pressure                   d = +0.303   (neg 129.25, pos 134.57)
  Serum Cholesterol                        d = +0.172   (neg 242.64, pos 251.47)
  Maximum Heart Rate Achieved              d = -0.912   (neg 158.38, pos 139.26)
  ST Depression (Exercise vs Rest)         d = +0.919   (neg 0.59, pos 1.57)
  Major Vessels Coloured by Fluoroscopy    d = +1.019   (neg 0.27, pos 1.14)


In [8]:
print(f"Cohort positive rate: {df[C.TARGET].mean():.3f}\n")
for feat, rates in eda["insights"]["categorical_positive_rates"].items():
    print(f"{C.FEATURE_BY_NAME[feat]['label']}:")
    for level, rate in rates.items():
        print(f"    {level:<32} {rate:.3f}")

Cohort positive rate: 0.459

Chest Pain Type:
    Typical angina                   0.304
    Atypical angina                  0.180
    Non-anginal pain                 0.209
    Asymptomatic                     0.729
Resting ECG Result:
    Normal                           0.371
    ST-T wave abnormality            0.750
    Left ventricular hypertrophy     0.541
Peak Exercise ST Segment Slope:
    Upsloping                        0.254
    Flat                             0.650
    Downsloping                      0.571
Thallium Stress Test Result:
    Normal                           0.223
    Fixed defect                     0.667
    Reversible defect                0.761
Sex:
    Female                           0.258
    Male                             0.553
Exercise-Induced Angina:
    No                               0.309
    Yes                              0.768
Fasting Blood Sugar above 120 mg/dl:
    No                               0.453
    Yes                         

## 5. Leakage control

Preprocessing lives inside the estimator pipeline. The check below shows the
scaler's statistics come from the training rows, not from the whole dataset.

In [9]:
from sklearn.model_selection import train_test_split
from src.preprocessing import build_preprocessor, encoded_feature_names

X, y = split_xy(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=C.TEST_SIZE, stratify=y, random_state=C.RANDOM_SEED)
print(f"train {len(X_train)} rows ({int(y_train.sum())} positive) | "
      f"test {len(X_test)} rows ({int(y_test.sum())} positive)")

prep = build_preprocessor(scale_numeric=True).fit(X_train)
scaler = prep.named_transformers_["num"].named_steps["scale"]
comparison = pd.DataFrame({
    "fitted_mean": scaler.mean_,
    "train_mean": X_train[C.NUMERIC_FEATURES].fillna(
        X_train[C.NUMERIC_FEATURES].median()).mean().to_numpy(),
    "full_dataset_mean": X[C.NUMERIC_FEATURES].fillna(
        X[C.NUMERIC_FEATURES].median()).mean().to_numpy(),
}, index=C.NUMERIC_FEATURES).round(4)
print()
print(comparison)
print(f"\nDesign matrix: {len(X.columns)} raw inputs -> "
      f"{len(encoded_feature_names(prep))} encoded columns")

train 242 rows (111 positive) | test 61 rows (28 positive)

          fitted_mean  train_mean  full_dataset_mean
age           54.5496     54.5496            54.4389
trestbps     130.9587    130.9587           131.6898
chol         249.8388    249.8388           246.6931
thalach      149.9628    149.9628           149.6073
oldpeak        0.9992      0.9992             1.0396
ca             0.6074      0.6074             0.6634

Design matrix: 13 raw inputs -> 22 encoded columns


## 6. Class imbalance

The training split is 1.18:1, which is mild. SMOTE was still evaluated - applied
**inside** each cross-validation fold, never to the whole dataset - so the
decision not to use it rests on a measurement.

In [10]:
from src.train import imbalance_study
study = imbalance_study(X_train, y_train, C.RANDOM_SEED)
pd.DataFrame({
    name: {metric: f"{vals['mean']:.4f} +/- {vals['std']:.4f}"
           for metric, vals in res.items()}
    for name, res in study["comparison"].items()
}).T

C:\Users\shahs\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,roc_auc,recall,f1
no_resampling,0.9069 +/- 0.0177,0.7656 +/- 0.0451,0.8136 +/- 0.0098
class_weight_balanced,0.9082 +/- 0.0184,0.8281 +/- 0.0542,0.8444 +/- 0.0264
smote_inside_folds,0.9086 +/- 0.0177,0.8190 +/- 0.0512,0.8347 +/- 0.0199


## 7. Baseline and the three comparison models

A quick untuned pass first, to see where each family starts. The tuned results
from the full run follow in section 8.

In [11]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from src.models import model_zoo

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=C.RANDOM_SEED)
rows = []
for name, spec in model_zoo().items():
    res = cross_validate(spec["pipeline"], X_train, y_train, cv=cv,
                         scoring=["roc_auc", "average_precision", "f1"], n_jobs=-1)
    rows.append({
        "model": name,
        "roc_auc": f"{res['test_roc_auc'].mean():.4f} +/- {res['test_roc_auc'].std():.4f}",
        "pr_auc": f"{res['test_average_precision'].mean():.4f}",
        "f1": f"{res['test_f1'].mean():.4f}",
        "notes": spec["notes"][:58] + "...",
    })
pd.DataFrame(rows).set_index("model")

,roc_auc,pr_auc,f1,notes
model,,,,
Logistic Regression,0.9068 +/- 0.0191,0.9021,0.8188,Regularised linear baseline; coefficients are ...
SVM (Linear),0.9023 +/- 0.0229,0.9067,0.8084,Maximum-margin linear separator; scaling is ma...
SVM (RBF),0.8860 +/- 0.0176,0.8828,0.8065,Non-linear kernel; C and gamma jointly control...
Random Forest,0.8829 +/- 0.0285,0.8842,0.7921,Bagged trees; depth and leaf size are the main...
XGBoost,0.8623 +/- 0.0236,0.8678,0.7444,Gradient-boosted trees with L2 regularisation;...


## 8. Tuned model comparison

Loaded from `models/metrics.json`, written by the full training run. CV columns
are the mean over 25 fits (5-fold x 5 repeats) on the training split; test
columns come from the 61 held-out records at the default 0.50 threshold, so
every model is compared on the same footing.

In [12]:
metrics = json.loads((C.MODELS_DIR / "metrics.json").read_text(encoding="utf-8"))
table = pd.DataFrame(metrics["comparison_table"]).set_index("model")
table[["cv_roc_auc_mean", "cv_roc_auc_std", "cv_pr_auc_mean", "test_accuracy",
       "test_precision", "test_sensitivity", "test_specificity", "test_f1",
       "test_roc_auc", "test_pr_auc"]].round(4)

,cv_roc_auc_mean,cv_roc_auc_std,cv_pr_auc_mean,test_accuracy,test_precision,test_sensitivity,test_specificity,test_f1,test_roc_auc,test_pr_auc
model,,,,,,,,,,
Logistic Regression,0.8976,0.0432,0.8938,0.8689,0.8125,0.9286,0.8182,0.8667,0.9589,0.9420
SVM (Linear),0.8950,0.0431,0.8935,0.8361,0.7647,0.9286,0.7576,0.8387,0.9556,0.9389
SVM (RBF),0.8968,0.0408,0.8932,0.8689,0.8125,0.9286,0.8182,0.8667,0.9578,0.9424
Random Forest,0.8934,0.0414,0.8935,0.8852,0.8387,0.9286,0.8485,0.8814,0.9470,0.9318
XGBoost,0.8954,0.0369,0.8966,0.8689,0.8333,0.8929,0.8485,0.8621,0.9437,0.9414


In [13]:
for name, detail in metrics["per_model_details"].items():
    print(f"{name}")
    print(f"  search     : {detail['search_strategy']}, scored on {detail['search_scoring']}")
    print(f"  best score : {detail['best_search_score']}")
    print(f"  best params: {detail['best_params']}\n")

Logistic Regression
  search     : GridSearchCV(24 combos), scored on roc_auc
  best score : 0.9085
  best params: {'clf__C': 0.5, 'clf__class_weight': 'balanced', 'clf__l1_ratio': 0.0}

SVM (Linear)
  search     : GridSearchCV(8 combos), scored on roc_auc
  best score : 0.9036
  best params: {'clf__estimator__C': 0.1, 'clf__estimator__class_weight': None}

SVM (RBF)
  search     : GridSearchCV(32 combos), scored on roc_auc
  best score : 0.9054
  best params: {'clf__estimator__C': 1.0, 'clf__estimator__class_weight': 'balanced', 'clf__estimator__gamma': 0.01}

Random Forest
  search     : RandomizedSearchCV(60 of 288 combos), scored on roc_auc
  best score : 0.8999
  best params: {'clf__n_estimators': 600, 'clf__min_samples_split': 5, 'clf__min_samples_leaf': 4, 'clf__max_features': 'sqrt', 'clf__max_depth': None, 'clf__class_weight': 'balanced'}

XGBoost
  search     : RandomizedSearchCV(60 of 384 combos), scored on roc_auc
  best score : 0.8997
  best params: {'clf__subsample': 0.8,

## 9. Final model selection

The rule was fixed before the numbers were in: rank on cross-validated ROC-AUC,
keep everything within one standard error, then separate the survivors on
calibration, interpretability, generalisation gap and cost.

In [14]:
sel = metrics["selection"]
print(sel["rule"], "\n")
print(f"best CV ROC-AUC : {sel['best_cv_roc_auc']}")
print(f"one-SE cut-offs : ROC-AUC >= {sel['roc_auc_one_se_cutoff']}, "
      f"PR-AUC >= {sel['pr_auc_one_se_cutoff']}")
print(f"shortlist       : {sel['shortlist']}")
print(f"selected        : {sel['selected']}\n")
pd.DataFrame(sel["ranking_within_shortlist"]).set_index("model")

1. Rank by mean cross-validated ROC-AUC on the training split and keep every model within one standard error of the best; at 242 training rows, differences smaller than that are not real. 2. Apply the same one-standard-error filter to PR-AUC, so a model that is clearly weaker on the positive class is dropped. 3. Among the survivors prefer the better-calibrated model, treating Brier differences below 0.01 as ties. 4. Break remaining ties on interpretability - a clinical screening aid has to be explainable - then on the train/test gap, then on fit cost. 

best CV ROC-AUC : {'model': 'Logistic Regression', 'mean': 0.8976, 'std': 0.0432, 'standard_error': 0.0086}
one-SE cut-offs : ROC-AUC >= 0.889, PR-AUC >= 0.8893
shortlist       : ['Logistic Regression', 'SVM (Linear)', 'SVM (RBF)', 'Random Forest', 'XGBoost']
selected        : Logistic Regression



,cv_roc_auc,cv_pr_auc,cv_brier,calibration_tie_group,interpretability_tier,generalisation_gap,fit_seconds
model,,,,,,,
Logistic Regression,0.8976,0.8938,0.1203,best,1,0.0337,0.04
SVM (Linear),0.8950,0.8935,0.1224,best,2,0.0294,0.05
XGBoost,0.8954,0.8966,0.1274,best,3,0.0552,0.13
SVM (RBF),0.8968,0.8932,0.1220,best,4,0.0247,0.06
Random Forest,0.8934,0.8935,0.1330,worse,3,0.0823,0.72


## 10. Threshold

0.50 is a default, not a decision. The operating point was chosen on
cross-validated *training* predictions: the highest-specificity threshold that
still reaches 85% sensitivity. The held-out test set played no part in it.

In [15]:
ta = metrics["threshold_analysis"]
pd.DataFrame([
    {"operating_point": "default 0.50", **{k: ta["default_metrics"][k] for k in
        ("threshold", "sensitivity", "specificity", "precision", "f1")}},
    {"operating_point": "best F1", **{k: ta["best_f1_metrics"][k] for k in
        ("threshold", "sensitivity", "specificity", "precision", "f1")}},
    {"operating_point": f"chosen (>= {ta['min_sensitivity_target']:.0%} sensitivity)",
     **{k: ta["chosen_metrics"][k] for k in
        ("threshold", "sensitivity", "specificity", "precision", "f1")}},
]).set_index("operating_point")

,threshold,sensitivity,specificity,precision,f1
operating_point,,,,,
default 0.50,0.50,0.8108,0.8855,0.8571,0.8333
best F1,0.51,0.8018,0.9008,0.8725,0.8357
chosen (>= 85% sensitivity),0.38,0.8649,0.7557,0.7500,0.8033


In [16]:
print("Held-out test performance of the selected model:\n")
for label, key in (("at the default 0.50", "final_test_metrics_default_threshold"),
                   (f"at the chosen {ta['chosen_threshold']}", "final_test_metrics_chosen_threshold")):
    m = metrics[key]
    print(f"  {label}:")
    for k in ("accuracy", "precision", "sensitivity", "specificity", "f1", "roc_auc", "pr_auc"):
        print(f"      {k:<12} {m[k]:.4f}")
    print(f"      confusion    {m['confusion_matrix']}\n")

Held-out test performance of the selected model:

  at the default 0.50:
      accuracy     0.8689
      precision    0.8125
      sensitivity  0.9286
      specificity  0.8182
      f1           0.8667
      roc_auc      0.9589
      pr_auc       0.9420
      confusion    {'tn': 27, 'fp': 6, 'fn': 2, 'tp': 26}

  at the chosen 0.38:
      accuracy     0.8197
      precision    0.7297
      sensitivity  0.9643
      specificity  0.6970
      f1           0.8308
      roc_auc      0.9589
      pr_auc       0.9420
      confusion    {'tn': 23, 'fp': 10, 'fn': 1, 'tp': 27}



## 11. Calibration

"Model-estimated probability" is only a useful phrase if the estimates track
observed frequencies. Brier score and expected calibration error are measured on
cross-validated training predictions; Platt scaling and isotonic regression were
both fitted inside the folds and compared.

In [17]:
cal = metrics["calibration"]
rows = []
for variant in ("raw", "sigmoid", "isotonic"):
    v = cal.get(variant, {})
    if "brier" in v:
        rows.append({"variant": variant, "brier": v["brier"],
                     "ECE": v["expected_calibration_error"], "roc_auc": v.get("roc_auc")})
print(f"lowest Brier: {cal['best_by_brier']}   |   applied: {cal['applied']}\n")
pd.DataFrame(rows).set_index("variant")

lowest Brier: raw   |   applied: raw



,brier,ECE,roc_auc
variant,,,
raw,0.1203,0.0638,0.9048
sigmoid,0.1228,0.0637,0.9049
isotonic,0.1213,0.0652,0.9000


## 12. Explainability

Three views. All of them describe the model's behaviour - none of them is
evidence that a feature *causes* disease.

In [18]:
expl = metrics["explainability"]
print("Permutation importance (drop in held-out ROC-AUC when the column is shuffled):")
display(pd.DataFrame(expl["permutation_importance"])[
    ["label", "mean_drop_in_roc_auc", "std"]].head(8).set_index("label"))

if expl.get("coefficients"):
    print("\nLargest coefficients of the deployed model (log-odds, standardised inputs):")
    display(pd.DataFrame(expl["coefficients"])[
        ["label", "coefficient", "odds_ratio"]].head(10).set_index("label"))

Permutation importance (drop in held-out ROC-AUC when the column is shuffled):


,mean_drop_in_roc_auc,std
label,,
Major Vessels Coloured by Fluoroscopy,0.08856,0.03390
Chest Pain Type,0.03084,0.01110
Thallium Stress Test Result,0.01392,0.00924
Peak Exercise ST Segment Slope,0.01259,0.00831
ST Depression (Exercise vs Rest),0.01032,0.00430
Maximum Heart Rate Achieved,0.00960,0.00765
Exercise-Induced Angina,0.00916,0.00312
Resting Blood Pressure,0.00725,0.00577



Largest coefficients of the deployed model (log-odds, standardised inputs):


,coefficient,odds_ratio
label,,
Major Vessels Coloured by Fluoroscopy,1.0278,2.7950
Sex,0.9841,2.6755
Thallium Stress Test Result: Normal,-0.7780,0.4593
Chest Pain Type: Asymptomatic,0.7568,2.1314
Thallium Stress Test Result: Reversible defect,0.6057,1.8326
Peak Exercise ST Segment Slope: Upsloping,-0.5580,0.5724
Chest Pain Type: Typical angina,-0.5557,0.5736
Chest Pain Type: Non-anginal pain,-0.5541,0.5746
Exercise-Induced Angina,0.4799,1.6160


In [19]:
shap_info = expl.get("shap_summary", {})
if shap_info.get("available"):
    print(f"Mean |SHAP| per feature - {shap_info['model']}, "
          f"{shap_info['rows_explained']} training records:")
    display(pd.DataFrame(shap_info["by_feature"]).head(8).set_index("label"))
else:
    print("SHAP summary unavailable:", shap_info.get("error"))

Mean |SHAP| per feature - XGBoost, 242 training records:


,feature,value
label,,
Thallium Stress Test Result,thal,0.73107
Chest Pain Type,cp,0.60294
Major Vessels Coloured by Fluoroscopy,ca,0.58559
Sex,sex,0.27413
Peak Exercise ST Segment Slope,slope,0.24607
ST Depression (Exercise vs Rest),oldpeak,0.21416
Age,age,0.15785
Maximum Heart Rate Achieved,thalach,0.15557


## 13. Error analysis

Which records the model gets wrong, and whether the mistakes share anything.

In [20]:
err = metrics["error_analysis"]
print("Test-set outcomes:", err["counts"], "\n")
for p in err["patterns"]:
    print(" -", p)
print("\n", err["cost_note"])

Test-set outcomes: {'true_positive': 27, 'true_negative': 23, 'false_positive': 10, 'false_negative': 1} 

 - Mean distance from the 0.38 threshold is 0.199 for wrong predictions versus 0.401 for correct ones - the model is measurably less certain when it errs.
 - 1 false negatives: mean age 60.0, mean vessels coloured 0.00, mean ST depression 3.00 mm (test-set averages: 54.0, 0.93, 1.20).
 - 10 false positives: mean age 55.8, mean vessels coloured 0.89, mean ST depression 0.69 mm (test-set averages: 54.0, 0.93, 1.20).

 A false negative sends a patient with significant narrowing away without follow-up; a false positive sends a healthy patient to further, sometimes invasive, testing. In screening the first error is normally the more serious one, which is why the operating threshold below is chosen to favour sensitivity rather than left at 0.5.


In [21]:
cases = pd.DataFrame([{
    "type": c["type"], "actual": c["true_label"],
    "model_probability": c["model_probability"],
    "distance_from_threshold": c["distance_from_threshold"],
    "strongest_influence": f"{c['top_features'][0]['label']} = {c['top_features'][0]['value']}",
} for c in err["cases"]])
cases

,type,actual,model_probability,distance_from_threshold,strongest_influence
0,false_negative,Positive,0.2921,0.0879,Chest Pain Type = Non-anginal pain
1,false_positive,Negative,0.6509,0.2709,Major Vessels Coloured by Fluoroscopy = 0
2,false_positive,Negative,0.3866,0.0066,Thallium Stress Test Result = Reversible defect
3,false_positive,Negative,0.4320,0.0520,Major Vessels Coloured by Fluoroscopy = 0
4,false_positive,Negative,0.5857,0.2057,Chest Pain Type = Typical angina
5,false_positive,Negative,0.7241,0.3441,Thallium Stress Test Result = Reversible defect
6,false_positive,Negative,0.6877,0.3077,Major Vessels Coloured by Fluoroscopy = 3
7,false_positive,Negative,0.4538,0.0738,Major Vessels Coloured by Fluoroscopy = 0
8,false_positive,Negative,0.5586,0.1786,Thallium Stress Test Result = Reversible defect
9,false_positive,Negative,0.9710,0.5910,Major Vessels Coloured by Fluoroscopy = 3


## 14. The inference pipeline

The same object the web application uses. Input is validated, transformed by the
saved preprocessing pipeline, scored, thresholded and explained.

In [22]:
from src.inference import get_predictor, validate_record, ValidationError

predictor = get_predictor()
record = {"age": 62, "sex": 0, "cp": 4, "trestbps": 140, "chol": 268, "fbs": 0,
          "restecg": 2, "thalach": 160, "exang": 0, "oldpeak": 3.6, "slope": 3,
          "ca": 2, "thal": 3}

result = predictor.predict(record).to_dict()
print(f"classification            : {result['classification']}")
print(f"model-estimated probability: {result['probability_percent']}%")
print(f"decision threshold         : {result['threshold_percent']}%")
print(f"confidence band            : {result['confidence_band']}\n")
print("Most influential inputs for this record:")
for item in result["explanation"]["items"]:
    print(f"  {item['label']:<42} {item['display_value']:<22} "
          f"{item['direction']:<10} {item['share']:.1%}")

classification            : Positive
model-estimated probability: 81.8%
decision threshold         : 38.0%
confidence band            : clearly on one side of the threshold

Most influential inputs for this record:
  Major Vessels Coloured by Fluoroscopy      2                      increases  34.0%
  Sex                                        Female                 decreases  14.0%
  Chest Pain Type                            Asymptomatic           increases  12.3%
  Thallium Stress Test Result                Normal                 decreases  11.9%
  ST Depression (Exercise vs Rest)           3.6 mm                 increases  9.8%
  Resting ECG Result                         Left ventricular hypertrophy increases  3.9%


In [23]:
# Validation rejects anything outside the documented schema, with a message per field.
for bad in ({**record, "age": 250}, {**record, "cp": 9}, {k: v for k, v in record.items() if k != "thal"}):
    try:
        validate_record(bad)
        print("accepted (unexpected)")
    except ValidationError as exc:
        print(exc.errors)

{'age': 'Age must be between 18 and 100 years.'}
{'cp': 'Chest Pain Type must be one of: 1 (Typical angina), 2 (Atypical angina), 3 (Non-anginal pain), 4 (Asymptomatic).'}
{'thal': 'Thallium Stress Test Result is required.'}


## 15. Where to go next

* `python -m src.train` re-runs everything above and rewrites `models/` and `outputs/`.
* `python -m uvicorn app.main:app --port 8000` serves the interface at
  <http://localhost:8000>, backed by these exact artefacts.
* `python -m pytest tests` runs the test suite.

---

**Medical disclaimer.** Educational use only. This project demonstrates
machine-learning classification on a public dataset. Model outputs are not
medical diagnoses, not medical advice, and not a substitute for evaluation by a
qualified healthcare professional.